In [ ]:
# install dependencies
%%capture
!pip install -q datasets sentence-transformers faiss-cpu transformers

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import re
import faiss
import pickle
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# load dataset and build corpus from contexts
dataset = load_dataset("pubmed_qa", "pqa_unlabeled")

def build_corpus(example):
    text = " ".join(example["context"]["contexts"])
    return {"text": text}

dataset = dataset.map(build_corpus)

corpus = dataset["train"]["text"]

In [ ]:
# clean text
def clean_text(t):
    t = t.lower()
    t = re.sub(r"\s+", " ", t)
    return t

corpus = [clean_text(t) for t in corpus]

In [ ]:
# analyze text lengths and print samples to get a sense of the data
corpus_subset = corpus[:50000] 

lengths = [len(text.split()) for text in corpus_subset]

plt.hist(lengths, bins=50)
plt.title("Text Length Distribution")
plt.show()

print("Avg:", np.mean(lengths))
print("Max:", np.max(lengths))

for i in range(2):
    print("\nSample:\n", corpus_subset[i][:300])

In [ ]:
# encode corpus with sentence transformermodel and convert to numpy array for faiss indexing
embedding_model = SentenceTransformer("all-MiniLM-L6-v2") # this is a smaller model that should be faster to encode with, but you can experiment with larger models for better performance

embeddings = embedding_model.encode(
    corpus_subset,
    batch_size=64,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

In [ ]:
# build faiss index and add vectors to it
dimension = embeddings.shape[1] # this should be 384 for the all-MiniLM-L6-v2 model

index = faiss.IndexFlatL2(dimension) 
index.add(embeddings)

print("Vectors:", index.ntotal)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Clustering with K-Means
num_clusters = 5  # You can adjust this based on the variance
kmeans = KMeans(n_clusters=num_clusters, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(embeddings)

# 2. Evaluation with Silhouette Score
# Note: Silhouette score can be slow on large datasets; using a sample of 10k for speed
sample_idx = np.random.choice(len(embeddings), 10000, replace=False)
score = silhouette_score(embeddings[sample_idx], cluster_labels[sample_idx])

print(f"Clusters Assigned: {np.unique(cluster_labels)}")
print(f"Silhouette Score (Unsupervised Proof): {score:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


# REDUCE DIMENSIONS (PCA)

pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings)

#  PLOT CLUSTERS
plt.figure(figsize=(10, 6))

scatter = plt.scatter(
    reduced_embeddings[:, 0],
    reduced_embeddings[:, 1],
    c=cluster_labels,
    cmap='viridis',
    s=10,
    alpha=0.7
)

# PLOT CENTROIDS

centroids_2d = pca.transform(kmeans.cluster_centers_)

plt.scatter(
    centroids_2d[:, 0],
    centroids_2d[:, 1],
    c='red',
    s=200,
    marker='X',
    label='Centroids'
)

plt.title("K-Means Clustering Visualization (PCA Reduced)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.legend()

plt.colorbar(scatter, label="Cluster ID")

plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

# SAMPLE DATA (IMPORTANT)
sample_size = 1000  # adjust if needed

sample_idx = np.random.choice(len(embeddings), sample_size, replace=False)
sample_embeddings = embeddings[sample_idx]


#  HIERARCHICAL CLUSTERING

linked = linkage(sample_embeddings, method='ward')


#  DENDROGRAM

plt.figure(figsize=(12, 6))

dendrogram(
    linked,
    truncate_mode='level',  # limits tree depth for clarity
    p=5,                   # show only top levels
    leaf_rotation=90,
    leaf_font_size=10,
    show_contracted=True
)

plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Sample Index")
plt.ylabel("Distance")

plt.show()

In [ ]:
# search function to query the index and return results
def search(query, k=3):
    query_vec = embedding_model.encode([query]).astype("float32")

    distances, indices = index.search(query_vec, k)

    results = []
    for i in range(k):
        idx = indices[0][i]
        results.append({
            "text": corpus_subset[idx],
            "score": float(distances[0][i])
        })

    return results

In [ ]:
# # I am loading a pre-trained FLAN-T5 large model and its tokenizer, then moving it to GPU if available so that text generation runs faster
model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
generation_model = generation_model.to(device)

In [ ]:
def generate_answer(query, text):
    # Build a prompt that gives the model context and instructs it to explain
    # the answer in simple language for a rural health worker.
    prompt = (
        f"Context: {text}\n\n"
        f"Task: You are a senior doctor. Explain the answer to a rural health worker using simple language.\n"
        f"Question: {query}\n"
        f"Simple Explanation:"
    )

    # Tokenize the prompt, truncate if needed, and move tensors to the model device.
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    # Generate the answer using the seq2seq model.
    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=150,          # allow up to 150 new tokens in the response
        num_beams=5,                 # use beam search for better quality
        no_repeat_ngram_size=3,      # reduce repeated phrases
        repetition_penalty=1.5,      # further discourage repetition
        length_penalty=1.2           # encourage slightly longer, more descriptive output
    )

    # Decode the generated token ids back into text.
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def generate_final_answer(query):
    # Retrieve top-k nearest neighbor passages from the FAISS index
    results = search(query, k=3)

    # Convert the L2 distance score to a similarity-like value
    best_similarity = 1 / (1 + results[0]['score'])

    # If the best match is reasonably close, use the retrieved context
    if best_similarity >= 0.45:
        # Combine the top retrieved texts into one prompt context
        combined_text = " ".join([r["text"] for r in results])[:1500]
        # Generate an answer using the retrieved evidence
        answer = generate_answer(query, combined_text)
        source = "Official Health Manual"
    else:
        # If the dataset match is weak, fall back to the model's general knowledge
        prompt = (
            f"Question: {query}\n\n"
            "Provide general health information and advice as a medical assistant. "
            "Always suggest consulting a doctor for persistent symptoms."
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7
        )
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
        source = "General AI Knowledge (Not in Manual)"

    return (
        f"[{source}]\n{answer}\n\n"
        "*DISCLAIMER: This is an AI assistant. Always consult a doctor before starting medication.*",
        results
    )

In [ ]:
# Define the query we want to ask the system
query = "What are the long-term effects of occupational back injuries?"

# Generate the final answer using the retrieval + generation pipeline
answer, results = generate_final_answer(query)

# Print the original query
print(" Query:", query)

# Print the generated answer
print("\n Answer:\n", answer)

# Print the retrieved evidence passages used for generating the answer
print("\n Evidence:\n")

for i, res in enumerate(results):
    # Convert the FAISS L2 distance score to a similarity-like value
    sim = 1 / (1 + res['score'])
    print(f"\nResult {i+1} (Similarity: {sim:.4f})")
    # Print the retrieved text passage
    print(res["text"][:])

In [ ]:
queries = [
    "fever treatment",
    "body pain remedies",
    "infection symptoms"
]

for q in queries:
    print("\n" + "="*60)
    print("Query:", q)
    ans, _ = generate_final_answer(q)
    print("Answer:", ans)

In [ ]:
# FAISS
faiss.write_index(index, "faiss_index.bin")

# corpus
with open("corpus.pkl", "wb") as f:
    pickle.dump(corpus_subset, f)

# embedding model
embedding_model.save("embedding_model")

# generation model
generation_model.save_pretrained("generation_model")
tokenizer.save_pretrained("generation_model")

In [ ]:
# import os
# import zipfile

# # Define exactly what you want to download
# files_to_zip = [
#     'faiss_index.bin',
#     'corpus.pkl',
#     'generation_model/', # The folder where you saved the model
#     'embedding_model/'
# ]

# zip_name = 'medassist_outputs.zip'

# with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
#     for item in files_to_zip:
#         if os.path.isfile(item):
#             zipf.write(item)
#         elif os.path.isdir(item):
#             for root, dirs, files in os.walk(item):
#                 for file in files:
#                     zipf.write(os.path.join(root, file))

# print(f"Created {zip_name} successfully!")